## Data Generation for Grocery Supply Chain
### Import Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import polars.selectors as cs
import os
import json
import fastparquet

import create_data_functions, weather_conditions

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

pl.set_random_seed(seed=56)

### Paths

In [2]:
# Define data paths
p = Path("data")

raw_data_path = p / 'raw'

processed_path = p / 'processed'

external_data_path = p / 'external'


In [3]:
# List of JSON filenames (without extension) to be loaded
arch_json = ['products','products_categories', 'suppliers']

# Dictionary to store the loaded JSON content
store_catalog = {}

# Loop through each filename, build the full path, and load the JSON data
for name in arch_json:
    file_path = raw_data_path / f"{name}.json"  # Construct full file path
    if file_path.exists():
        with file_path.open("r", encoding="utf-8") as f:     # Open the JSON file
            store_catalog[name] = json.load(f)                        # Load and store the data under its name
    else:
        print(f"Notice: File {file_path} was not found.")

# Catalog Information

In [4]:
# Calculate the total number of products in the store catalog
total_products = len(store_catalog["products"])
# Calculate the total number of suppliers in the store catalog
total_suppliers = len(store_catalog["suppliers"])

# Create unique supplier IDs with suffix 'S'
suppliers_id = create_data_functions.create_IDs(total_suppliers, suffix='S')

# Initialize a random number generator with a fixed seed for reproducibility
rng = np.random.default_rng(seed=43)

# Randomly select 15 unique suppliers to be considered "top suppliers"
suppliers_top = rng.choice(list(set(store_catalog['suppliers'].keys())), 15, replace=False)

# Create a LazyFrame of products with product names as a column
catalog_lazy = (
    pl.DataFrame(store_catalog["products"])  # Convert products dictionary to DataFrame
    .transpose(include_header=True)          # Transpose to align product data correctly
    .lazy()                                  # Convert to LazyFrame for deferred execution
    .rename({"column": "product"})           # Rename column to 'product'
    .unnest("column_0")                      # Expand nested column values
    .with_columns(
        pl.Series(
            "product_id",
            create_data_functions.create_IDs(total_products, suffix='P')  # Generate product IDs
        ))
    .select([                                # Select relevant product attributes
        "product_id",
        "product",
        "category",
        "sub_category",
        "shelf_life_days",
        "maximum_days_on_sale",
        "seasonality",
        "storage_recommendation",
        "unit_of_measurement"
        ])

    # Join product data with supplier data
    .join(
        pl.from_dicts(store_catalog["suppliers"])   # Convert suppliers dictionary to DataFrame
        .transpose(include_header=True)             # Transpose to align supplier data correctly
        .unnest("column_0")                         # Expand nested column values
        .with_columns([
            pl.Series("supplier_id", suppliers_id), # Add supplier IDs
            pl.Series("supplier_rating", np.random.randint(1, 6, size=total_suppliers)).cast(pl.UInt8)  # Random ratings
        ])
        .lazy()                                     # Convert to LazyFrame
        .rename({
            "products": "product",                  # Rename 'products' to 'product'
            "column": "supplier"                    # Rename 'column' to 'supplier'
        })
        .explode("product")                         # Expand product lists per supplier
        .select([                                   # Select relevant supplier attributes
            "supplier_id", "supplier_rating", "supplier", "product", "distance_km", "moq"
        ])
    ,
    on="product",                                   # Join on product field
    how="inner"                                     # Inner join to match products with suppliers
    )
)

In [5]:
# View Catalog
catalog_lazy.head().collect()

product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq
str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64
"""1459079|P""","""Spinach""","""Fresh Foods""","""Vegetables""",5,2,"[""April"", ""May"", … ""September""]","""Refrigerated""","""lb""","""1715504|S""",2,"""GreenFields Co.""",127,150
"""1504689|P""","""Lettuce""","""Fresh Foods""","""Vegetables""",7,3,"[""January"", ""February"", … ""December""]","""Refrigerated""","""unit""","""1715504|S""",2,"""GreenFields Co.""",127,150
"""1044985|P""","""Kale""","""Fresh Foods""","""Vegetables""",5,2,"[""January"", ""February"", … ""December""]","""Refrigerated""","""lb""","""1715504|S""",2,"""GreenFields Co.""",127,150
"""1159259|P""","""Cabbage""","""Fresh Foods""","""Vegetables""",14,7,"[""January"", ""February"", … ""December""]","""Refrigerated""","""unit""","""1715504|S""",2,"""GreenFields Co.""",127,150
"""1816015|P""","""Broccoli""","""Fresh Foods""","""Vegetables""",7,3,"[""October"", ""November"", … ""March""]","""Refrigerated""","""lb""","""1715504|S""",2,"""GreenFields Co.""",127,150


## Meteorological Data for Supply Chain Management

In [6]:
# Set the path to the external weather data source
# Source: https://bdmep.inmet.gov.br/

# Set the path to the weather CSV file
# archive_csv = external_data_path + 'dados_83967_D_2015-01-01_2025-09-18.csv'
archive_csv = external_data_path / 'dados_B807_D_2022-12-07_2025-09-22.csv'

# Rename columns to clear and descriptive English names
columns_name = {
    "Data Medicao": "measurement_date",
    "PRECIPITACAO TOTAL, DIARIO (AUT)(mm)": "daily_total_precipitation_mm",
    "TEMPERATURA MAXIMA, DIARIA (AUT)(°C)": "daily_maximum_temperature_c",
    "TEMPERATURA MINIMA, DIARIA (AUT)(°C)": "daily_minimum_temperature_c",
    "VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)": "daily_average_wind_speed_mps"
}

In [7]:
# Read the CSV file into a LazyFrame for efficient query planning
weather_lazy = (
    pl.scan_csv(
        archive_csv,
        separator=";",          # Use semicolon as delimiter
        decimal_comma=True,     # Interpret comma as decimal separator
        skip_rows=10,           # Skip metadata/header rows at the top
        try_parse_dates=True    # Attempt automatic date parsing
    )
    # Rename columns to clear and descriptive English names
    .rename(columns_name)

    # Drop empty column (created by trailing delimiter in CSV)
    .drop("")

    # Replace string "null" with actual None values and cast to Float32
    .with_columns([
        pl.when(pl.col(c) == "null")
        .then(None)             # Convert "null" string to None
        .otherwise(pl.col(c))   # Keep original value otherwise
        .cast(pl.Float32)       # Cast to float for numeric operations
        .alias(c)               # Preserve column name
        for c in columns_name.values() if c != "measurement_date"
    ])

    # Remove rows where all values are null (fully missing records)
    .filter(~pl.all_horizontal(pl.all().is_null()))

    # Remove the first two rows (index 0 and 1) to clean dataset
    .slice(2, None)

    .with_columns(
        # Create binary flags indicating missing values for each variable
        pl.col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"),
        pl.col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"),
        pl.col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"),
        pl.col("daily_average_wind_speed_mps").is_null().alias("wind_missing"),

        # Impute missing values using forward fill (propagate last valid observation forward)
        pl.col("daily_total_precipitation_mm").fill_null(strategy="forward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="forward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="forward"),
    )

    .with_columns(
        # Impute missing values using backward fill (propagate next valid observation backward)
        pl.col("daily_total_precipitation_mm").fill_null(strategy="backward"),
        pl.col("daily_maximum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_minimum_temperature_c").fill_null(strategy="backward"),
        pl.col("daily_average_wind_speed_mps").fill_null(strategy="backward"),
    )

    # Select final set of columns: original values + missing flags
    .select(
        [
            'measurement_date',                # Keep the measurement date column
            'daily_total_precipitation_mm',    # Original precipitation values
            'precipitation_missing',           # Binary flag for missing precipitation
            'daily_maximum_temperature_c',     # Original maximum temperature values
            'max_temp_missing',                # Binary flag for missing max temperature
            'daily_minimum_temperature_c',     # Original minimum temperature values
            'min_temp_missing',                # Binary flag for missing min temperature
            'daily_average_wind_speed_mps',    # Original wind speed values
            'wind_missing'                     # Binary flag for missing wind speed
        ]
    ) 
)

In [8]:
# Collect the weather data into a DataFrame
df_weather = weather_lazy.collect()

# Find the minimum measurement date
date_min = df_weather["measurement_date"].min()

# Find the maximum measurement date
date_max = df_weather["measurement_date"].max()

# Generate the expected sequence of dates from min to max, with daily intervals
expect_seq = pl.date_ranges(date_min, date_max, interval="1d", eager=True).explode()

# Get the unique measurement dates from the dataset and sort them
exist_date = df_weather["measurement_date"].unique().sort()

# Check if the existing dates match the expected sequence (detect gaps)
gaps = not exist_date.equals(expect_seq)

# Print whether there are date gaps
print(f"Have date gaps? {gaps}")

Have date gaps? False


In [9]:
# Get schema without collecting the full dataset
print(weather_lazy.collect_schema())

Schema([('measurement_date', Date), ('daily_total_precipitation_mm', Float32), ('precipitation_missing', Boolean), ('daily_maximum_temperature_c', Float32), ('max_temp_missing', Boolean), ('daily_minimum_temperature_c', Float32), ('min_temp_missing', Boolean), ('daily_average_wind_speed_mps', Float32), ('wind_missing', Boolean)])


In [10]:
# Polars optimization
print(weather_lazy.explain())

simple π 9/9 ["measurement_date", ... 8 other columns]
   WITH_COLUMNS:
   [col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
     WITH_COLUMNS:
     [col("daily_total_precipitation_mm").is_null().alias("precipitation_missing"), col("daily_maximum_temperature_c").is_null().alias("max_temp_missing"), col("daily_minimum_temperature_c").is_null().alias("min_temp_missing"), col("daily_average_wind_speed_mps").is_null().alias("wind_missing"), col("daily_total_precipitation_mm").fill_null_with_strategy(), col("daily_maximum_temperature_c").fill_null_with_strategy(), col("daily_minimum_temperature_c").fill_null_with_strategy(), col("daily_average_wind_speed_mps").fill_null_with_strategy()] 
      SLICE[offset: 2, len: 18446744073709551615]
        FILTER [([([([(col("measurement_date").is_null()

### Define Weather Severity Levels

In [11]:
# Apply the weather classification function to the cleaned DataFrame to generate severity and category labels
weather_analyser = weather_conditions.PolarsWeatherConditions(weather_lazy)
weather_severity_lazy = (
    weather_analyser.classify_weather()
    .select([
            'measurement_date',
            'temperature_classification',
            'precipitation_classification',
            'wind_classification', 
            'weather_severity'
        ])
    .rename({"measurement_date": "received_date"})
)

In [12]:
# Show 10 samples rows of the DataFrame
weather_severity_lazy.head(3).collect()

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity
date,str,str,str,str
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate"""


In [13]:
# Collects and returns the schema definition 
weather_severity_lazy.collect_schema()

Schema([('received_date', Date),
        ('temperature_classification', String),
        ('precipitation_classification', String),
        ('wind_classification', String),
        ('weather_severity', String)])

# Realistic supply chain modeling based on Weather, Product and Seasonality

In [14]:
# Determine the number of samples based on the length of the weather DataFrame
n_samples = len(weather_severity_lazy.collect())

# Calculate how many times we need to replicate the weather data to reach ~300,000 rows
quantity_rows = 1000
multiply_rows = quantity_rows // n_samples + 1
n_total = multiply_rows * n_samples

# Replicate the fixed weather conditions multiply_rows times
weather_replicated_lazy = pl.concat([weather_severity_lazy] * multiply_rows)

# Sample products with replacement to match the total number of rows
catalog_sampled_lazy = catalog_lazy.collect().sample(n=n_total, with_replacement=True)

# Horizontally merge weather and product data row-by-row, then sort by received_date
merged_prod_weather_lazy = (
    weather_replicated_lazy
    .collect()
    .hstack(catalog_sampled_lazy)
    .sort("received_date")
    .with_columns(
        (pl.col("received_date").dt.strftime("%B").is_in(pl.col("seasonality")).alias("in_season"))
    )
).lazy()

In [15]:
merged_prod_weather_lazy.show(3)

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,in_season
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,bool
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1562525|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1036823|S""",1,"""ValleyFresh Farms""",85,100,false
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1605792|P""","""Pasta""","""Pantry""","""Grains & Rice""",730,180,[],"""Room Temperature""","""lb""","""1737959|S""",5,"""Pasta Paradise""",130,100,false
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1469171|P""","""Salmon Fillet""","""Fresh Foods""","""Seafood""",2,1,[],"""Refrigerated""","""lb""","""1491867|S""",3,"""Seafood Select""",195,28,false


In [16]:
print(merged_prod_weather_lazy.collect_schema())

Schema([('received_date', Date), ('temperature_classification', String), ('precipitation_classification', String), ('wind_classification', String), ('weather_severity', String), ('product_id', String), ('product', String), ('category', String), ('sub_category', String), ('shelf_life_days', Int64), ('maximum_days_on_sale', Int64), ('seasonality', List(String)), ('storage_recommendation', String), ('unit_of_measurement', String), ('supplier_id', String), ('supplier_rating', UInt8), ('supplier', String), ('distance_km', Int64), ('moq', Int64), ('in_season', Boolean)])


## Generate data about holidays, weekdays of the year.

In [17]:
prod_seasonality_lazy = (
    create_data_functions.day_classification_lazy(df=merged_prod_weather_lazy, col="received_date", country="br")
)

## Generate data for stock quantities and sales volumes.

In [ ]:
# Generate sales demand
data_stock_sales_lazy = create_data_functions.classify_grocery_demand_polars(df=prod_seasonality_lazy, columns_date="received_date", country="br")

# Generate sales volume
data_stock_sales_lazy = create_data_functions.simulate_sales_volume_polars(df= data_stock_sales_lazy)

# Generate sales volume
data_stock_sales_lazy = create_data_functions.estimate_delivery_days_polars(df= data_stock_sales_lazy)

# Generate Min_Max Stock
data_stock_sales_lazy = create_data_functions.min_max_stock_polars(df = data_stock_sales_lazy)

### Generate average sales

In [ ]:
data_stock_sales_lazy = data_stock_sales_lazy.with_columns(
    # Calculate the average sales volume per supplier and product
    pl.col("sales_volume").mean().over(["supplier_id", "product_id"])
    # Cast the result to Int16 for compact storage and consistency
    .cast(pl.Int16)
    # Assign the result to a new column named "avg_sales"
    .alias("avg_sales")
)


In [30]:
data_stock_sales_lazy.show()

received_date,temperature_classification,precipitation_classification,wind_classification,weather_severity,product_id,product,category,sub_category,shelf_life_days,maximum_days_on_sale,seasonality,storage_recommendation,unit_of_measurement,supplier_id,supplier_rating,supplier,distance_km,moq,in_season,is_holiday,day_classification,is_weekend,sales_demand,sales_volume,delivery_days,min_stock,max_stock,avg_sales
date,str,str,str,str,str,str,str,str,i64,i64,list[str],str,str,str,u8,str,i64,i64,bool,bool,str,bool,str,i64,u16,i16,i16,i16
2022-12-09,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1562525|P""","""Tomato""","""Fresh Foods""","""Vegetables""",7,3,"[""June"", ""July"", … ""September""]","""Room Temperature""","""lb""","""1036823|S""",1,"""ValleyFresh Farms""",85,100,false,false,"""Weekday""",false,"""High""",142,3,462,677,154
2022-12-10,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1605792|P""","""Pasta""","""Pantry""","""Grains & Rice""",730,180,[],"""Room Temperature""","""lb""","""1737959|S""",5,"""Pasta Paradise""",130,100,false,false,"""Weekday""",false,"""High""",61,4,242,342,60
2022-12-11,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1469171|P""","""Salmon Fillet""","""Fresh Foods""","""Seafood""",2,1,[],"""Refrigerated""","""lb""","""1491867|S""",3,"""Seafood Select""",195,28,false,false,"""Weekday""",false,"""High""",9,5,43,73,8
2022-12-12,"""Warm""","""No precipitation""","""Gentle to Fresh Breeze""","""Moderate""","""1357337|P""","""Milk""","""Dairy & Alternatives""","""Dairy""",7,3,[],"""Refrigerated""","""unit""","""1555797|S""",5,"""DairyPure Inc.""",50,80,false,false,"""Weekday""",false,"""Normal""",108,3,330,432,110
2022-12-13,"""Mild to Temperate""","""Moderate Rain""","""Gentle to Fresh Breeze""","""Moderate""","""1312215|P""","""Sugar""","""Pantry""","""Baking Supplies""",730,180,[],"""Room Temperature""","""lb""","""1510109|S""",5,"""Wholesale Warehouse""",25,300,false,false,"""Weekday""",false,"""Normal""",15,4,44,337,11


### Create stock distribution

In [ ]:
def create_stock_distribution_vectorized(stock_min, stock_max, seed: int=None, 
                                         prob_stock: list=[0.12, 0.28, 0.60],
                                         prob_extreme: list=[0.68, 0.27, 0.05]):
    """
    Generates a vector of stock quantities based on probabilistic conditions: 'out of stock', 'overstocked', or 'normal',
    applied element-wise to input vectors.

    Parameters:
    ----------
    stock_min : pandas.Series
        Series of minimum stock quantities for each item.
    stock_max : pandas.Series
        Series of maximum stock quantities for each item.
    seed : int, optional
        Seed for the random number generator to ensure reproducibility.
    prob_stock : list of float, optionalpl.DataFrame | pl.LazyFrame
        Probabilities for each stock condition: ['out', 'over', 'normal'] respectively.
        Default is [0.12, 0.28, 0.60].
    prob_extreme : list of float, optional
        Probabilities for selecting intervals within 'out' and 'over' conditions.
        Default is [0.68, 0.27, 0.05].

    Returns:
    -------
    numpy.ndarray
        Array of generated stock quantities for each item.

    Stock Conditions:
    -----------------
    - 'normal': Random integer between stock_min[i] and stock_max[i].
    - 'over': Value above stock_max[i] using a multiplier from a probabilistic interval.
    - 'out': Value below stock_min[i] using a divider from a probabilistic interval.

    Example:
    --------
    >>> import pandas as pd
    >>> stock_min = pd.Series([50, 30, 20])
    >>> stock_max = pd.Series([100, 80, 60])
    >>> create_stock_distribution_vectorized(stock_min, stock_max, seed=42)
    array([  0,  94,  44])  # (actual values may vary depending on condition and seed)
    """

    # Initialize random number generator with optional seed
    rng = np.random.default_rng(seed=seed)
    n = len(stock_min)

    # Define possible stock conditions
    stock_condition = ['out', 'over', 'normal']

    # Randomly assign a condition to each item
    conditions = rng.choice(stock_condition, size=n, p=prob_stock)

    # Initialize result array
    results = np.zeros(n, dtype=int)

    # Process each item based on its assigned condition
    for i, condition in enumerate(conditions):
        if condition == 'normal':
            # Generate stock within normal range
            results[i] = rng.integers(stock_min.iloc[i], stock_max.iloc[i] + 1)

        elif condition == 'over':
            # Define intervals for overstock multipliers
            multipliers_intervals = [(1.05, 1.15), (1.16, 1.30), (1.31, 1.70)]
            # Select an interval based on extreme probabilities
            chosen_interval = rng.choice(multipliers_intervals, p=prob_extreme)
            # Generate multiplier and apply to stock_max
            multiplier = rng.uniform(chosen_interval[0], chosen_interval[1])
            results[i] = int(np.ceil(stock_max.iloc[i] * multiplier))

        elif condition == 'out':
            # Define intervals for out-of-stock dividers
            dividers_intervals = [(0.05, 0.15), (0.16, 0.30), (0.31, 0.70)]
            # Select an interval based on extreme probabilities
            chosen_interval = rng.choice(dividers_intervals, p=prob_extreme)
            # Generate divider and apply to stock_min
            divider = rng.uniform(chosen_interval[0], chosen_interval[1])
            results[i] = int(np.floor(stock_min.iloc[i] * divider))

    return results
